In [ ]:
import random
from datasets import load_dataset, get_dataset_config_names
import numpy as np
import time

# ==============================================================================
# 🌟 데이터셋 정보: nayohan/Maths-College-ko 🌟
# 의미: 한국어로 된 수학(Maths) 관련 문제와 그 풀이 과정을 학습한 데이터셋입니다.
# 설명: 이 데이터셋은 인공지능(AI)에게 "이 문제(instruction)를 받았을 때, 이렇게 풀이해야 해(output)"라는 형태의 수학적 추론 과정을 가르치기 위해 사용됩니다.
# 우리의 목표: 이 데이터가 실제로 어떤 퀄리티의 수학 학습 자료로 사용될 수 있는지, 초보자의 관점에서 분석하고 맛보기입니다!
# ==============================================================================

# ⚙️ 설정 변수
DATASET_NAME = "nayohan/Maths-College-ko"
SPLIT_TO_LOAD = "train"
SAMPLE_COUNT = 10  # 실습을 위해 상위 10개 샘플만 가져옵니다. (데이터셋이 크기 때문에!)
# ==============================================================================

# 1. 사용 가능한 Config 이름 확인 (최신 패턴 적용)
print("=" * 80)
print("💡 Step 1. 데이터셋 Config 확인 및 로드 준비")
print("=" * 80)

try:
    configs = get_dataset_config_names(DATASET_NAME)
    print(f"✅ 사용 가능한 Config 목록: {configs}")
    
    # 첫 번째 config를 기본값으로 사용합니다.
    selected_config = configs[0]
except Exception as e:
    print(f"ℹ️ Config 확인 중 오류가 발생했거나 별도의 Config가 없습니다: {e}")
    selected_config = None

# 2. 데이터 로드 (스트리밍 우선 시도)
print("\n✨ 2. 데이터셋을 메모리로 로드합니다...")

dataset = None
try:
    # 🚀 Attempt 1: 스트리밍 모드로 로드 시도 (대용량 데이터에 최적!)
    print("   >>> [시도 1/2] 스트리밍(streaming=True) 모드로 데이터 로드를 시도합니다...")
    dataset = load_dataset(DATASET_NAME, name=selected_config, split=SPLIT_TO_LOAD, streaming=True)
    print("   ✅ 성공! 스트리밍 모드로 데이터셋을 로드했습니다. 메모리 효율 최고!")
    
except Exception as e:
    # 🛑 Fallback: 스트리밍이 실패하거나 환경 문제 발생 시
    print(f"   ❌ 경고: 스트리밍 모드 로드 실패 ({e.__class__.__name__}).")
    print("   ➡️ 대체: 일반(Non-streaming) 모드로 소량만 다운로드하여 진행합니다.")
    try:
        dataset = load_dataset(DATASET_NAME, name=selected_config, split=SPLIT_TO_LOAD, streaming=False)
    except Exception as inner_e:
        print(f"🚨 치명적 오류: 데이터를 로드할 수 없습니다. {inner_e}")
        exit()

print(f"\n[📊 데이터셋 요약]: 데이터셋 이름: {DATASET_NAME} | 로드 분할(Split): {SPLIT_TO_LOAD}")
print(f"   (주의) 데이터셋 전체 크기는 매우 크므로, 상위 {SAMPLE_COUNT}개만 샘플링하여 실습합니다.")

# 3. 샘플링된 데이터 준비 (기술적 요구사항 준수 패턴 사용)
# .take() 메서드를 사용하여 상위 K개만 가져옵니다.
print("\n==============================================================================")
print("🚀 Step 3. 데이터 샘플링 및 순회 준비")
print("==============================================================================")

if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)
    print(f"✅ 스트리밍 모드 감지: 상위 {SAMPLE_COUNT}개 샘플을 순회할 준비를 합니다.")
    sample_dataset_iterator = dataset.take(SAMPLE_COUNT)
else:
    # 일반 데이터셋 (Dataset)
    print(f"✅ 일반 모드 감지: 상위 {SAMPLE_COUNT}개 샘플을 목록으로 저장합니다.")
    sample_dataset = dataset.select(range(SAMPLE_COUNT))
    sample_dataset_iterator = iter(sample_dataset)

# 샘플 데이터를 리스트로 변환 (반복문에서 안정적으로 접근하기 위해)
sample_data_list = list(sample_dataset_iterator)

if not sample_data_list:
    print("🛑 에러: 샘플 데이터를 가져오는 데 실패했습니다. 스크립트를 종료합니다.")
    exit()

print(f"🌟 🎉 성공! 총 {len(sample_data_list)}개의 샘플 데이터를 준비했습니다. 이제 '지식 탐험가'가 되어 분석을 시작해봅시다!")


# 4. 창의적 초보자 실습: 지식 추출 및 Prompt Engineering 시뮬레이션
print("\n" + "=" * 80)
print("✨ 4. AI 프롬프트 설계 체험: 수학 지식 추출 실습 (지식 탐험가 모드)")
print("=" * 80)

print("🚀 오늘의 목표: AI가 이해하기 쉽도록 '문제'와 '정답 과정'을 분리하여 구조화하는 법을 배웁니다.")
print("⭐ 데이터 구조: 'instruction' (질문) -> 'output' (정답/풀이 과정)\n")


def analyze_and_display_sample(sample_data_list, index):
    """개별 샘플을 분석하여 사용자에게 친절하게 설명해주는 함수"""
    sample = sample_data_list[index]
    
    instruction = sample['instruction']
    output = sample['output']
    
    print(f"▶️ [{index + 1}/{len(sample_data_list)}] 예제 분석 (Prompt Engineering 시뮬레이션):")
    
    # 💡 주석 처리된 부분은 개발자가 참고하는 메모입니다.
    # -------------------------------------------------------------------
    # ❓ instruction: 이것이 AI에게 던져줄 '문제'입니다. (Input Prompt)
    # 📝 output: 이것이 AI가 따라 해야 할 '정답 과정'입니다. (Target Answer)
    # -------------------------------------------------------------------

    print(f"  📚 [INPUT PROMPT (지시문)]: {instruction[:40]}...")
    print("     -> (AI에게 던지는 질문 역할을 합니다. 구체적일수록 좋습니다!)")
    print("-" * 50)
    print(f"  ✅ [TARGET OUTPUT (기대 답변)]: {output[:80]}...")
    print("     -> (AI가 최종적으로 생성해야 할, 완벽하게 구조화된 해답입니다!)")
    print("\n✨ 튜터의 코멘트: 이 데이터 쌍은 '문제→해답'의 완벽한 학습 쌍입니다. 이 구조를 반복 학습할수록 AI의 수학 추론 능력이 길러집니다!")


# 5. 실습 실행 (반복문 사용)
for i, sample_data in enumerate(sample_data_list):
    analyze_and_display_sample(sample_data, i)

# 6. 결론 및 다음 단계 안내
print("\n" + "=" * 80)
print("🎉 ✨ 실습 완료! 수고하셨습니다, 미래의 AI 개발자님! ✨ 🎉")
print("==============================================================================")
print("""
[🎓 튜터가 드리는 다음 학습 Tip]
1. 데이터 필터링: 만약 이 데이터 중 '초등 수학'만 보고 싶다면, instruction에 '초등학교'라는 키워드가 포함된 데이터만 골라낼 수 있습니다 (필터링 실습).
2. 평가 지표: 데이터가 잘 로드되었으니, 다음은 이 데이터로 '평가 데이터셋'을 만들고, 모델의 성능을 측정하는 연습을 해봅시다.
3. 확장 가능성: 이 원리(instruction -> output)는 수학 외에도 '영어 문장 변환', '코딩 문제 풀이' 등 모든 QA(Question-Answering) 영역에 적용될 수 있어요!
""".replace("\n", "\n"))